# Stage 3B — Direct Q-Concentration Optimization and Causal Validation

This notebook reloads frozen outputs produced by `python -m scripts.stage3b`. It does not train, select checkpoints, evaluate BER, or use Stage 2E.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd
ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'outputs' / 'stage3b').exists())
OUT = ROOT / 'outputs' / 'stage3b'
diagnostics = json.loads((OUT / 'stage3b_diagnostics.json').read_text())
diagnostics['outcome'], diagnostics['interpretation']

## Frozen Q-only intervention

For each fixed atomic bank, $Q_W=M^{-1}\sum_m |W^H\Delta(\nu_m)\Pi(\tau_m)W|^{\odot2}$. Arm A minimizes $1-\eta_{24}^Q$. Arm B adds $0.4$ times one minus the mean concentration of the weakest 10% of rows. Hard top-24 positions come from detached $Q$; gradients pass only through selected values. Checkpoints use the independent Q-validation bank only.

In [ ]:
history = pd.read_csv(OUT / 'training_validation_history.csv')
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for arm, axis in zip(('mean', 'robust'), axes):
    for name, rows in history[history.arm == arm].groupby('initialization', sort=False):
        axis.plot(rows.step, rows.validation_eta, label=name)
    axis.set(title=arm.title(), xlabel='step', ylabel='validation eta_24^Q'); axis.grid(alpha=.3); axis.legend();

In [ ]:
causal = pd.read_csv(OUT / 'causal_intervention_summary.csv')
causal[['arm', 'initialization', 'selected_step', 'validation_eta_initial', 'validation_eta_selected', 'heldout_eta_q_delta', 'heldout_fixed_retained_energy_mean_delta', 'heldout_fixed_analytic_mse_mean_delta', 'qpsk_q_relative_change', 'qam16_q_relative_change']]

In [ ]:
structure = pd.read_csv(OUT / 'structural_communication_metrics.csv')
structure[structure.group.isin(['stage3b', 'principal'])][['waveform', 'eta_q', 'row_p05', 'effective_support_mean', 'k90', 'k95', 'k99', 'k995', 'fixed_retained_energy_mean', 'fixed_analytic_mse_mean', 'q_to_c_spearman']]

In [ ]:
communication = pd.read_csv(OUT / 'communication_summary.csv')
communication[(communication.snr_db == 18) & (communication.support_rule == 'Q') & communication.waveform.isin(list(causal.waveform) + ['Learned', 'OFDM', 'OTFS', 'AFDM'])][['modulation', 'waveform', 'ber', 'fixed_vs_oracle_relative_penalty']]

In [ ]:
aligned = pd.read_csv(OUT / 'aligned_basis_similarity.csv')
pairwise = pd.read_csv(OUT / 'solution_pairwise_similarity.csv')
display(aligned.sort_values(['waveform', 'mean_matched_squared_overlap'], ascending=[True, False]).groupby('waveform').head(1))
pairwise[['left', 'right', 'mean_matched_squared_overlap', 'aligned_chordal_distance']]

## Interpretation

**Outcome A, qualified.** Both Q-only arms rescue both Haar starts and improve held-out concentration, fixed-support retained energy, analytic MSE, QPSK BER, and 16-QAM BER. Robust row-tail protection is consistent but modest. The solutions do not collapse to a tested classical or Learned basis. Q-concentration is therefore causally supported as a primary finite-representation design property in this frozen setting, but the remaining QPSK gap to authoritative Learned shows that it is not a complete communication objective.